# Assignment 2: Local Features, Matching and Panorama Stitching
**MSAI6330 Computer Vision**

Goal: match local invariant features between two overlapping photos, estimate a homography with RANSAC, and stitch a panorama. We run the whole pipeline twice, once with SIFT and once with raw intensity patches, then compare.

| # | Section | Rubric marks |
|---|---|---|
| 1 | Feature detection | 10 |
| 2 | Feature description (SIFT and 11x11 patch) | 15 |
| 3-4 | Descriptor matching and putative match selection | 15 |
| 5 | Match visualization | 10 |
| 6 | Homography estimation with RANSAC | 20 |
| 7 | Panorama creation | 15 |
| 8 | Comparison and conclusion | 15 (shared with organization) |

**Conventions.** Image 1 is `left.jpg` and is the reference frame. Image 2 is `right.jpg`. The homography `H` maps image 2 coordinates into image 1 coordinates. Keypoints are stored as `N x 2` arrays of `(x, y)`.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams["figure.dpi"] = 110
rng = np.random.default_rng(0)          # fixed seed so RANSAC results are reproducible

IMG_DIR = Path("images")
left_bgr = cv2.imread(str(IMG_DIR / "left.jpg"))
right_bgr = cv2.imread(str(IMG_DIR / "right.jpg"))
assert left_bgr is not None and right_bgr is not None, "run the notebook from the repo root so images/ is found"

left = cv2.cvtColor(left_bgr, cv2.COLOR_BGR2RGB)
right = cv2.cvtColor(right_bgr, cv2.COLOR_BGR2RGB)
gray_l = cv2.cvtColor(left_bgr, cv2.COLOR_BGR2GRAY)
gray_r = cv2.cvtColor(right_bgr, cv2.COLOR_BGR2GRAY)
print("OpenCV", cv2.__version__, "| image size (h, w):", left.shape[:2], right.shape[:2])

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].imshow(left); ax[0].set_title("left.jpg (image 1, reference)")
ax[1].imshow(right); ax[1].set_title("right.jpg (image 2, to be warped)")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

## 1. Feature Detection
> **Requirement:** load the two images and detect feature points in both using Harris corners or blob detection.

_TODO: briefly explain your method and what you observe in the results._

In [ ]:
def harris_keypoints(gray, k=0.04, block=3, rel_thresh=0.01, nms_size=9, border=8, max_pts=1500):
    """TODO: Harris corners. Steps: cv2.cornerHarris -> threshold relative to max -> non-maximum
    suppression (compare to cv2.dilate) -> drop border pixels -> keep strongest max_pts.
    Return (pts as Nx2 float32 array of (x, y), response map)."""
    raise NotImplementedError


# TODO: Harris keypoints for both images (harris_l, harris_r)
# TODO: SIFT keypoints + descriptors for both images with cv2.SIFT_create().detectAndCompute
#       -> kp_l, sift_desc_l, kp_r, sift_desc_r, and sift_l, sift_r as Nx2 arrays

# TODO: visualize the detected points on both images

## 2. Feature Description
> **Requirement:** extract descriptors for the keypoints, testing (a) SIFT and (b) an intensity-based descriptor made from a local patch (e.g. 11x11) flattened to a 1D vector.

_TODO: briefly explain your method and what you observe in the results._

In [ ]:
def patch_descriptors(gray, pts, size=11, normalize=True):
    """TODO: for every keypoint cut a size x size window, flatten to 1D, optionally normalise
    (subtract mean, divide by std). Return an N x size*size float32 array."""
    raise NotImplementedError


# TODO: patch descriptors for harris_l / harris_r
# TODO: print descriptor shapes for both types and show a few example patches

## 3. Descriptor Matching
> **Requirement:** compute pairwise Euclidean distances between descriptors from the two images.

_TODO: briefly explain your method and what you observe in the results._

In [ ]:
def pairwise_dist(A, B):
    """TODO: return the len(A) x len(B) table of Euclidean distances (vectorised, no double loop)."""
    raise NotImplementedError


# TODO: build a dict `data` with, for each of "SIFT" and "Patch": pts1, pts2, d1, d2 and the distance table D
# TODO: sanity-check pairwise_dist against a brute-force loop on a small slice

## 4. Putative Match Selection
> **Requirement:** select candidate matches using a threshold or the top matches with the smallest distances, and explain the strategy.

_TODO: briefly explain your method and what you observe in the results._

In [ ]:
def select_matches(D, ratio=0.8, mutual=True):
    """TODO: for each row find the best and second-best column; keep if d_best < ratio * d_second
    (and, if mutual, if the best column's own best row is this row).
    Return (pairs as M x 2 int array of (i, j), their distances)."""
    raise NotImplementedError


# TODO: choose a strategy / threshold per descriptor, store data[name]["pairs"], print the counts,
#       and EXPLAIN your choice in the markdown cell above

## 5. Match Visualization
> **Requirement:** place the two images side by side and draw lines between corresponding keypoints.

_TODO: briefly explain your method and what you observe in the results._

In [ ]:
def draw_matches(img1, img2, pts1, pts2, pairs, ax, title, inlier_mask=None, max_lines=80):
    """TODO: build a side-by-side canvas (np.hstack / manual paste), draw a line for every pair.
    Remember to shift image-2 x coordinates by img1.shape[1]. Colour by inlier_mask if given."""
    raise NotImplementedError


# TODO: show the putative matches for SIFT and for the patch descriptor

## 6. Homography Estimation Using RANSAC
> **Requirement:** implement RANSAC for the homography; report the number of inliers, the average squared reprojection error of the inliers, and a visualization of the inlier correspondences.

_TODO: briefly explain your method and what you observe in the results._

In [ ]:
def apply_H(H, pts):
    """TODO: apply a 3x3 homography to Nx2 points (homogeneous coordinates, then divide by w)."""
    raise NotImplementedError


def sq_reproj_error(H, src, dst):
    """TODO: squared Euclidean distance between H*src and dst, one value per point."""
    raise NotImplementedError


def ransac_homography(src, dst, n_iter=3000, eps=3.0, rng=rng):
    """TODO: RANSAC loop. Sample 4 pairs -> cv2.getPerspectiveTransform -> count inliers
    (squared error < eps**2) -> keep the best -> refit on all inliers with cv2.findHomography(..., 0).
    Return (H, inlier_mask, squared_errors)."""
    raise NotImplementedError


# TODO: run RANSAC for SIFT and Patch matches (src = image-2 points, dst = image-1 points)
# TODO: report the inlier count and the mean squared reprojection error of the inliers
# TODO: visualize the inlier correspondences

## 7. Panorama Creation
> **Requirement:** create a new image big enough to hold the panorama and composite the two images into it (average in the overlap, or use pixels from one image).

_TODO: briefly explain your method and what you observe in the results._

In [ ]:
def make_panorama(img_ref, img_mov, H, max_pixels=40_000_000):
    """TODO: 1) transform the corners of img_mov with H, 2) bounding box with img_ref's corners,
    3) translation T so the box starts at (0, 0), 4) cv2.warpPerspective(img_mov, T @ H, size),
    5) paste img_ref at the offset, 6) average (or copy one image) where both have pixels.
    Return the panorama, or None if H is unusable."""
    raise NotImplementedError


# TODO: build and show the panorama for both descriptor types

## 8. Comparison: SIFT vs Intensity-Based Descriptor
> **Requirement:** compare the descriptors by number of putative matches, number of RANSAC inliers, reprojection error and final panorama quality.

_TODO: briefly explain your method and what you observe in the results._

In [ ]:
# TODO: print a comparison table: putative matches, RANSAC inliers, inlier ratio, mean squared
#       reprojection error, for SIFT and Patch.
# TODO: show both panoramas one above the other.

## Conclusion
> **Requirement:** a short conclusion summarizing the main findings.

_TODO: write 5-8 sentences: which descriptor did better on each measure and why (invariance vs discriminability), what the panorama looks like, and one limitation._